# Judge scale, and whether examples help

Reviewer 2 asks for different families, different scales and a few-shot setting;
Reviewer 3 asks for additional sizes or families. The family is answered by the
Falcon run. This covers the other two.

Three configurations, one run, so that every number is produced by the same loader,
the same prompt builder and the same scoring code, all fetched from the released tag:
Qwen 2.5 3B zero-shot and Qwen 2.5 14B zero-shot, which move scale with the family
held fixed, and Qwen 2.5 7B few-shot, which moves the prompt with the model held
fixed at the one the paper reports at kappa = 0.115.

The few-shot examples come from the 30-trajectory pilot, never from the 200 being
scored. Drawing them from the evaluation corpus would leak labels into the prompt.

The 14B runs last: it is the configuration most likely to exhaust two T4s, and a
failure there must not cost the two results before it.

In [ ]:
import ast, gc, json, re, time, urllib.request
import torch
from collections import Counter

BASE = ("https://raw.githubusercontent.com/achenna-source/"
        "icv-agent-observability/neucom-d-26-17269-r1/")
fetch = lambda p: urllib.request.urlopen(BASE + p, timeout=90).read().decode("utf-8")

ns = {}
exec(compile(fetch("code/judge.py"), "judge.py", "exec"), ns)
build_prompt, parse_reply, GEN = ns["build_prompt"], ns["parse_reply"], ns["GENERATION_KWARGS"]
mns = {}
exec(compile(fetch("code/metrics.py"), "metrics.py", "exec"), mns)
kappa_lenient, cluster_bootstrap = mns["kappa_lenient"], mns["cluster_bootstrap"]

rows = [json.loads(l) for l in fetch("data/trajectories_200.jsonl").splitlines() if l.strip()]
pilot = [json.loads(l) for l in fetch("data/trajectories_pilot_30.jsonl").splitlines() if l.strip()]
_kb = json.loads(fetch("data/knowledge_base.json"))
kb, cities_absent = _kb["facts"], _kb["populations_absent_by_design"]
steps_of = lambda r: ast.literal_eval(r["trajectory"]) if isinstance(r["trajectory"], str) else r["trajectory"]

human = [r["label_human"] for r in rows]
valid = set(human)
clusters = [r.get("cluster_id", r["id"]) for r in rows]
print("evaluation corpus %d | pilot %d | facts %d | absent %d"
      % (len(rows), len(pilot), len(kb), len(cities_absent)))
print("gpus: %d" % torch.cuda.device_count())

In [ ]:
# Les exemples few-shot viennent du pilote. Trois modes distincts, pris dans l'ordre
# du fichier pour qu'aucune selection ne soit faite sur la reponse attendue.
WANT = ["S", "F3", "F7"]
shots = []
for code in WANT:
    for r in pilot:
        if r["label_human"] == code:
            shots.append(r)
            break
print("few-shot examples drawn from the pilot: %s"
      % [(r["id"], r["label_human"]) for r in shots])

def shot_block():
    out = []
    for r in shots:
        p = build_prompt({"task": r["task"], "trajectory": steps_of(r)}, kb, cities_absent)
        body = p.split("TRAJECTOIRE :", 1)[1].split("Reponds UNIQUEMENT", 1)[0].strip()
        out.append("EXEMPLE :\nTRAJECTOIRE :\n%s\nReponse : {\"code\":\"%s\"}"
                   % (body, r["label_human"]))
    return "\n\n".join(out)

SHOTS = shot_block()
print("\nfew-shot block: %d characters" % len(SHOTS))

def make_prompt(r, few_shot):
    p = build_prompt({"task": r["task"], "trajectory": steps_of(r)}, kb, cities_absent)
    if not few_shot:
        return p
    head, tail = p.split("TRAJECTOIRE :", 1)
    return head + SHOTS + "\n\nTRAJECTOIRE :" + tail

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def run(model_id, few_shot, tag):
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16,
                                                 device_map="auto")
    model.eval()
    print("\n[%s] %s loaded in %.0f s" % (tag, model_id, time.time() - t0))
    preds = []
    t0 = time.time()
    for i, r in enumerate(rows):
        msgs = [{"role": "user", "content": make_prompt(r, few_shot)}]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        enc = tok(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, pad_token_id=tok.eos_token_id, **GEN)
        preds.append(parse_reply(tok.decode(out[0][enc["input_ids"].shape[1]:],
                                            skip_special_tokens=True)))
        if (i + 1) % 50 == 0:
            print("   %3d/200  %.0f s" % (i + 1, time.time() - t0))
    del model, tok
    gc.collect(); torch.cuda.empty_cache()
    return [p if p else "UNPARSED" for p in preds]

def score(pred, tag):
    agree = sum(1 for a, b in zip(human, pred) if a == b)
    k = kappa_lenient(human, pred, valid)
    items = [(c, (h, p)) for c, h, p in zip(clusters, human, pred)]
    lo, hi, _ = cluster_bootstrap(items, lambda d: kappa_lenient(
        [h for h, _ in d], [p for _, p in d], valid))
    print("[%s] agreement %d/200 = %.1f%% | kappa %.3f | 95%% CI [%.3f, %.3f]"
          % (tag, agree, 100.0 * agree / 200, k, lo, hi))
    print("      codes assigned: %s" % dict(Counter(pred).most_common(6)))
    return {"tag": tag, "agreement": agree, "kappa": round(k, 3),
            "ci": [round(lo, 3), round(hi, 3)], "preds": pred}

RESULTS = []

In [ ]:
p = run("Qwen/Qwen2.5-3B-Instruct", False, "3B zero-shot")
RESULTS.append(score(p, "3B zero-shot"))
json.dump(RESULTS, open("judge_sweep.json", "w"))

In [ ]:
p = run("Qwen/Qwen2.5-7B-Instruct", True, "7B few-shot")
RESULTS.append(score(p, "7B few-shot"))
json.dump(RESULTS, open("judge_sweep.json", "w"))

In [ ]:
try:
    p = run("Qwen/Qwen2.5-14B-Instruct", False, "14B zero-shot")
    RESULTS.append(score(p, "14B zero-shot"))
except Exception as e:
    print("14B did not run: %s" % str(e)[:300])
json.dump(RESULTS, open("judge_sweep.json", "w"))

In [ ]:
print("=== for the manuscript ===")
print("%-16s %10s %8s %18s" % ("configuration", "agreement", "kappa", "95% interval"))
print("%-16s %9.1f%% %8.3f %18s" % ("7B zero-shot", 30.0, 0.115, "published"))
for r in RESULTS:
    print("%-16s %9.1f%% %8.3f   [%+.3f, %+.3f]"
          % (r["tag"], 100.0 * r["agreement"] / 200, r["kappa"], r["ci"][0], r["ci"][1]))
print("%-16s %9.1f%% %8.3f   [%+.3f, %+.3f]" % ("Falcon3 7B", 28.5, 0.038, -0.037, 0.118))